# LAVA: Local Genetic Correlation Analysis
**CHD (Septal Defects) × ASD**

## 1. Setup

In [ ]:
shell_exec <- function(cmd, ...) {
  out <- system(cmd, intern = TRUE, ...)
  cat(paste0(out, collapse = "\n"))
}

In [ ]:
install.packages("R.utils", quiet = TRUE)

if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install(c("snpStats", "GenomicRanges", "GenomeInfoDb", "BiocGenerics", "rtracklayer"),
                     update = FALSE, ask = FALSE)

if (!require("remotes", quietly = TRUE)) install.packages("remotes")
remotes::install_github("josefin-werme/LAVA", quiet = TRUE)

if (!require("pacman", quietly = TRUE)) install.packages("pacman")
pacman::p_load_gh("trinker/qdapTools")

In [ ]:
library(data.table)
library(tidyverse)
library(LAVA)
library(GenomicRanges)
library(rtracklayer)
library(qdapTools)

## 2. GWAS Summary Statistics Preparation

### 2.1 Congenital Heart Disease (Septal Defects) — FinnGen Meta-analysis

In [ ]:
chd_gwas <- fread("/content/METAANALYSIS1_Septal.TBL")

In [ ]:
# Total N: FinnGen cases + external cases + controls
chd_gwas[, N := 451549 + 2184 + 458440]

chd_gwas <- chd_gwas[, .(SNP = MarkerName,
                          A1 = toupper(Allele1),
                          A2 = toupper(Allele2),
                          N,
                          B  = Effect,
                          SE = StdErr,
                          P  = `P-value`)]

chd_gwas[, Z := B / SE]

In [ ]:
# Annotate chromosomal position from 1000 Genomes EUR reference panel
bim <- fread("/content/g1000_eur/g1000_eur.bim",
             col.names = c("CHR", "SNP", "CM", "BP", "A1", "A2"))

chd_gwas <- bim[, .(SNP, BP, CHR)][chd_gwas, on = "SNP"]

rm(bim)

In [ ]:
fwrite(chd_gwas, file = "/content/chd_summaries.lava.gz", sep = "\t")
rm(chd_gwas)

### 2.2 Autism Spectrum Disorder — PGC GWAS

In [ ]:
shell_exec('wget --user-agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0" \
           -O /content/asdPGC.zip \
           "https://figshare.com/ndownloader/articles/14671989/versions/1"')

shell_exec("unzip -o /content/asdPGC.zip -d /content/")

In [ ]:
asd_gwas <- fread("/content/iPSYCH-PGC_ASD_Nov2017.gz")

asd_gwas[, N := 18382 + 27969]
asd_gwas[, B := log(OR)]
asd_gwas[, Z := B / SE]

# Allele coding convention: swap A1/A2 to match LAVA input format
asd_gwas[, c("A1", "A2") := .(A2, A1)]

asd_gwas <- asd_gwas[, .(SNP, CHR, A1, A2, N, Z, B, P, SE, BP)]

In [ ]:
fwrite(asd_gwas, file = "/content/asd_summaries.lava.gz", sep = "\t")
rm(asd_gwas)

## 3. LAVA Analysis: CHD × ASD

### 3.1 Study Configuration

In [ ]:
study_info <- data.table(
  phenotype = c("CHD", "ASD"),
  cases     = c(2184 + 687, 18382),
  controls  = c(451549 + 457753, 27969),
  filename  = c("/content/chd_summaries.lava.gz",
                "/content/asd_summaries.lava.gz")
)

fwrite(study_info,
       "/content/phenotypes_prevalence_CHDasd.LAVA.txt",
       sep = "\t", na = "NA", quote = FALSE)

### 3.2 LD Loci Partitioning

In [ ]:
system("git clone https://github.com/cadeleeuw/lava-partitioning.git")

### 3.3 Select Loci Blocks

<!-- PLACEHOLDER: paste loci selection code here -->

In [ ]:
ld_blocks <- fread("/content/lava-partitioning/LAVA_s2500_m25_f1_w200.blocks")

In [ ]:
shell_exec("wget ftp://ftp.ensembl.org/pub/grch37/release-87/gtf/homo_sapiens/Homo_sapiens.GRCh37.87.gtf.gz")

In [ ]:
shell_exec("gunzip Homo_sapiens.GRCh37.87.gtf.gz")

In [ ]:
gwas_list <-
  setNames(
    object = list(
      fread("/content/asd_summaries.lava.gz"),
      fread("/content/chd_summaries.lava.gz")
    ),
    nm = c("CHD", "ASD")
  )

In [ ]:
ref <- import("/content/Homo_sapiens.GRCh37.87.gtf")
ref <- ref %>% keepSeqlevels(c(1:22), pruning.mode = "coarse")
ref <- ref[ref$type == "gene"]

In [ ]:
colnames(gwas_list[[2]])

In [ ]:
# Filter for only genome-wide significant loci and convert to granges
gr_list <-
  gwas_list %>%
  lapply(., function(gwas){
    gwas %>%
      dplyr::filter(P < 5e-8) %>%
      GenomicRanges::makeGRangesFromDataFrame(
        .,
        keep.extra.columns = TRUE,
        ignore.strand = TRUE,
        seqinfo = NULL,
        seqnames.field = "CHR",
        start.field = "BP",
        end.field = "BP",
        na.rm=TRUE
      )
  })

In [ ]:
ld_blocks <-
  ld_blocks %>%
  dplyr::rename_with(
    .fn = stringr::str_to_upper,
    .cols = everything()
  ) %>%
  dplyr::mutate(
    LOC = dplyr::row_number()
  ) %>%
  dplyr::select(LOC, everything())

In [ ]:
ld_blocks_gr <-
  ld_blocks %>%
  GenomicRanges::makeGRangesFromDataFrame(
    .,
    keep.extra.columns = TRUE,
    ignore.strand = TRUE,
    seqinfo = NULL,
    seqnames.field = "chr",
    start.field = "start",
    end.field = "stop"
  )

In [ ]:
overlap_list <-
  gr_list %>%
  lapply(., function(gr){
    GenomicRanges::findOverlaps(gr, ld_blocks_gr, type = "within") %>%
      as_tibble()
  })

In [ ]:
loci <-
  ld_blocks %>%
  dplyr::slice(
    overlap_list %>%
      qdapTools::list_df2df(col1 = "gwas") %>%
      .[["subjectHits"]] %>%
      unique()
    ) %>%
  dplyr::arrange(LOC)

In [ ]:
# Generate df of GWAS loci that overlap which LD blocks
overlap_df_list <- vector(mode = "list", length = length(gr_list))

for(i in 1:length(gr_list)){

  gr <- gr_list[[i]]

  names(overlap_df_list)[i] <- names(gr_list)[i]

  if(names(gr_list)[i] == c("mdd")){

    overlap_df_list[[i]] <-
      gr %>%
      as_tibble() %>%
      dplyr::mutate(
        BETA = NA,
        OR = NA
      ) %>%
      dplyr::select(seqnames, start, end, SNP, SNP, A1, A2, B, Z, P, N, SE)


  } else if(names(gr_list)[i] == c("scz")){

    overlap_df_list[[i]] <-
      gr %>%
      as_tibble() %>%
      dplyr::mutate(
        BETA = NA,
        logOdds = NA
      ) %>%
      dplyr::select(seqnames, start, end, SNP, A1, A2, B, Z, P, N, SE)

  } else{

    overlap_df_list[[i]] <-
      gr %>%
      as_tibble() %>%
      dplyr::mutate(
        OR = NA,
        logOdds = NA
      ) %>%
      dplyr::select(seqnames, start, end, SNP, A1, A2, B, Z, P, N, SE)
 # 'SNP''CHR''A1''A2''N''Z''B''P''BP'; Add SE?
  }

  overlap_df_list[[i]] <-
    overlap_df_list[[i]] %>%
    dplyr::rename_with(
      ~ stringr::str_c("GWAS", .x, sep = "_")
    ) %>%
    dplyr::slice(overlap_list[[i]]$queryHits) %>%
    dplyr::bind_cols(
      ld_blocks_gr %>%
        as_tibble() %>%
        dplyr::rename_with(
          ~ stringr::str_c("LD", .x, sep = "_")
        ) %>%
        dplyr::slice(overlap_list[[i]]$subjectHits)
    ) %>%
    dplyr::select(-contains("strand")) %>%
    dplyr::rename_with(
      ~ stringr::str_replace(.x,
                             pattern = "seqnames",
                             replacement = "CHR"),
      .col = dplyr::ends_with("seqnames"))

}

overlap_df <-
  overlap_df_list %>%
  qdapTools::list_df2df(col1= "GWAS")

In [ ]:
# Generate df of associated GWAS and genes that overlap investigated LD blocks
loci_gr <-
  loci %>%
  GenomicRanges::makeGRangesFromDataFrame(
    .,
    keep.extra.columns = TRUE,
    ignore.strand = TRUE,
    seqinfo = NULL,
    seqnames.field = "CHR",
    start.field = "START",
    end.field = "STOP"
  )

overlap <-
  GenomicRanges::findOverlaps(loci_gr, ref) %>%
  tibble::as_tibble()

loci_genes <-
  tibble::tibble(
    locus = loci_gr[overlap$queryHits]$LOC,
    chr = loci_gr[overlap$queryHits] %>% GenomeInfoDb::seqnames() %>% as.character(),
    locus_start = loci_gr[overlap$queryHits] %>% BiocGenerics::start(),
    locus_end = loci_gr[overlap$queryHits] %>% BiocGenerics::end(),
    gene_id = ref[overlap$subjectHits]$gene_id,
    gene_name =
      ref[overlap$subjectHits]$gene_name %>%
      stringr::str_replace_all("-", "_") %>%
      stringr::str_replace_all("\\.", "_")
  ) %>%
  dplyr::group_by(locus, chr, locus_start, locus_end) %>%
  dplyr::summarise(
    n_overlapping_genes = n(),
    overlapping_gene_id = list(gene_id),
    overlapping_gene_name = list(gene_name)
      )

loci_genes_gwas_df <-
  loci_genes %>%
  dplyr::inner_join(
    overlap_df %>%
      dplyr::distinct(GWAS, LD_LOC) %>%
      dplyr::group_by(LD_LOC) %>%
      dplyr::summarise(
        n_assoc_gwas = n(),
        assoc_gwas = list(str_to_upper(GWAS))
        ),
    by = c("locus" = "LD_LOC")
  )

In [ ]:
system("mkdir /content/filtered_loci")

In [ ]:
out_dir <- "filtered_loci"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

write_delim(
  loci,
  delim = "\t",
  file = file.path(out_dir, "gwas_filtered.loci")
  )
saveRDS(
  overlap_df,
  file = file.path(out_dir, "gwas_filtered_loci.rds")
  )
saveRDS(
  loci_genes_gwas_df,
  file = file.path(out_dir, "gwas_filtered_loci_w_genes.rds")
)

In [ ]:
remove(gwas_list)

### 3.4 Reference Panel (1000 Genomes EUR)

In [ ]:
shell_exec('wget -O /content/g1000_eur.zip \
            "https://vu.data.surfsara.nl/index.php/s/VZNByNwpD8qqINe/download"')

shell_exec("unzip -o /content/g1000_eur.zip -d /content/g1000_eur/")

### 3.5 Process LAVA Input

In [ ]:
lava_input <- LAVA::process.input(
  input.info.file    = "/content/phenotypes_prevalence_CHDasd.LAVA.txt",
  sample.overlap.file = NULL,
  ref.prefix         = "/content/g1000_eur/g1000_eur",
  phenos             = c("CHD", "ASD")
)

In [ ]:
ls(lava_input)
ls(lava_input$sum.stats)
lava_input$info
head(lava_input$ref$bim)

In [ ]:
loci   <- LAVA::read.loci("/content/filtered_loci/gwas_filtered.loci")
n_loci <- nrow(loci)

### 3.6 Run Univariate and Bivariate Tests

In [ ]:
# Bonferroni-corrected significance threshold for univariate filtering
univ_threshold <- 0.05 / n_loci

# Progress checkpoints at every 5% of loci
progress_idx <- ceiling(quantile(1:n_loci, probs = seq(0.05, 1, 0.05)))

univ_results <- vector("list", n_loci)
bivar_results <- vector("list", n_loci)

for (i in seq_len(n_loci)) {

  if (i %in% progress_idx)
    message(sprintf("Progress: %s", names(progress_idx[which(progress_idx == i)])))

  locus <- LAVA::process.locus(loci[i, ], lava_input)

  if (!is.null(locus)) {

    locus_meta <- data.frame(
      locus  = locus$id,
      chr    = locus$chr,
      start  = locus$start,
      stop   = locus$stop,
      n_snps = locus$n.snps,
      n_pcs  = locus$K
    )

    locus_out <- LAVA::run.univ.bivar(locus, univ.thresh = univ_threshold)

    univ_results[[i]] <- dplyr::bind_cols(locus_meta, locus_out$univ)

    if (!is.null(locus_out$bivar))
      bivar_results[[i]] <- dplyr::bind_cols(locus_meta, locus_out$bivar)
  }
}

## 4. Save Results

In [ ]:
phenotypes <- c("CHD", "ASD")
out_dir    <- "/content/results"
dir.create(out_dir, showWarnings = FALSE)

output_tag <- paste(phenotypes, collapse = ":")

saveRDS(univ_results,  file = file.path(out_dir, paste0(output_tag, ".univ.lava.rds")))
saveRDS(bivar_results, file = file.path(out_dir, paste0(output_tag, ".bivar.lava.rds")))

In [ ]:
univ_df  <- dplyr::bind_rows(purrr::compact(univ_results))
bivar_df <- dplyr::bind_rows(purrr::compact(bivar_results))

write.csv(univ_df,  file.path(out_dir, "univar_results.csv"),  row.names = FALSE)
write.csv(bivar_df, file.path(out_dir, "bivar_results.csv"),   row.names = FALSE)